In [76]:
import numpy as np
import torch
import torch.nn as nn

import torch.nn.functional as F
import time

from torch.utils.data import DataLoader

from torchvision import transforms, datasets

In [78]:
class Inception(nn.Module):
    def __init__(self, in_channels, c1, c2, c3, c4):
        
        super(Inception, self).__init__()
        
        # 线路1，1x1
        self.p1_1 = nn.Conv2d(in_channels, c1, kernel_size=1)
        
        # 线路2,1x1后接3x3
        self.p2_1 = nn.Conv2d(in_channels, c2[0], kernel_size=1)
        self.p2_2 = nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1)
        
        # 线路3，1x1后接5x5，
        self.p3_1 = nn.Conv2d(in_channels, c3[0], kernel_size=1)
        self.p3_2 = nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2)
        
        # 线路4，最大池化后接1x1
        self.p4_1 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.p4_2 = nn.Conv2d(in_channels, c4, kernel_size=1)

    def forward(self, x):
        
        #p1 = F.relu(self.p1_1(X))
        p1 = self.p1_1(x)
        p1 = torch.relu(p1)

        #p2 = F.relu(self.p2_2(F.relu(self.p2_1(X))))
        p2 = self.p2_1(x)
        p2 = torch.relu(p2)
        p2 = self.p2_2(p2)
        p2 = torch.relu(p2)

        #p3 = F.relu(self.p3_2(F.relu(self.p3_1(X))))
        p3 = self.p3_1(x)
        p3 = torch.relu(p3)
        p3 = self.p3_2(p3)
        p3 = torch.relu(p3)

        #p4 = F.relu(self.p4_2((self.p4_1(X)))
        p4 = self.p4_1(x)
        p4 = self.p4_2(p4)
        p4 = torch.relu(p4)
        
        return torch.cat((p1, p2, p3, p4), dim=1)

In [81]:
b1 = nn.Sequential(nn.Conv2d(3, 128, kernel_size=7, stride=2, padding=3),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

b2 = nn.Sequential(nn.Conv2d(128, 128, kernel_size=1),
                    nn.ReLU(),
                    nn.Conv2d(128, 192, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

b3 = nn.Sequential(Inception(192, 64, (96, 128), (16, 32), 32),
                    Inception(256, 128, (128, 192), (32, 96), 64),
                    nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

b4 = nn.Sequential(Inception(480, 192, (96, 208), (16, 48), 64),
                    Inception(512, 160, (112, 224), (24, 64), 64),
                    Inception(512, 128, (128, 256), (24, 64), 64),
                    Inception(512, 112, (144, 288), (32, 64), 64),
                    Inception(528, 256, (160, 320), (32, 128), 128),
                    nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

b5 = nn.Sequential(Inception(832, 256, (160, 320), (32, 128), 128),
                    Inception(832, 384, (192, 384), (48, 128), 128),
                    nn.AdaptiveAvgPool2d((1, 1)),
                    nn.Flatten())

netg = nn.Sequential(b1, b2, b3, b4, b5,
                    
                     nn.Linear(1024, 10))


In [83]:
X = torch.rand(size=(1, 3, 224, 224))
for layer in netg:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t', X.shape)

Sequential output shape:	 torch.Size([1, 128, 56, 56])
Sequential output shape:	 torch.Size([1, 192, 28, 28])
Sequential output shape:	 torch.Size([1, 480, 14, 14])
Sequential output shape:	 torch.Size([1, 832, 7, 7])
Sequential output shape:	 torch.Size([1, 1024])
Linear output shape:	 torch.Size([1, 10])


In [86]:
data_transform = transforms.Compose([
    transforms.Resize(256),     #resize图片
    transforms.CenterCrop(224), #随机裁剪
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) #标准化
])

# 加载数据集
train_sets = datasets.CIFAR10(root='cifar_10', 
                     train=True, 
                     download=True, 
                     transform=data_transform)

test_sets = datasets.CIFAR10(root='cifar_10', 
                    train=False, 
                    download=True, 
                    transform=data_transform)

batch_size = 64

train_loader = torch.utils.data.DataLoader(dataset=train_sets, 
                                           batch_size=batch_size, 
                                           shuffle=True)  #将数据打乱

test_loader = torch.utils.data.DataLoader(dataset=test_sets, 
                                          batch_size=batch_size, 
                                          shuffle=True)


In [88]:
def evaluate_accuracy(data_iter, net):
    acc_sum, n = 0.0, 0
    
    net.eval()
    for X, y in data_iter:
        X = X.to(device)  
        y = y.to(device)
        acc_sum += (net(X).argmax(dim=1) == y).float().sum().item()
        n += y.shape[0]
    return acc_sum / n

In [89]:
def train(net, train_iter, test_iter, batch_size, optimizer, device, num_epochs):
    
    net = net.to(device)
    print("training on ", device)
    loss = torch.nn.CrossEntropyLoss()
    batch_count = 0
    for epoch in range(num_epochs):
        
        train_l_sum, train_acc_sum, n, start = 0.0, 0.0, 0, time.time()
        
        net.train()
        for X, y in train_iter:
            X = X.to(device)
            y = y.to(device)
            
            y_hat = net(X)
            l = loss(y_hat, y)
            
            optimizer.zero_grad()
            l.backward()
            optimizer.step()
            
            train_l_sum += l.cpu().item()
            train_acc_sum += (y_hat.argmax(dim=1) == y).sum().cpu().item()
            n += y.shape[0]
            batch_count += 1

        test_acc = evaluate_accuracy(test_iter, net)
        print('epoch %d, loss %.4f, train acc %.3f, test acc %.3f, time %.1f sec'
              % (epoch + 1, train_l_sum / batch_count, train_acc_sum / batch_count, test_acc, time.time() - start))

In [94]:
torch.cuda.empty_cache()

In [96]:
train_iter, test_iter = train_loader, test_loader

lr = 0.001

num_epochs = 10

optimizer = torch.optim.Adam(netg.parameters(), lr)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [98]:
# 启动
train(netg, train_iter, test_iter, batch_size, optimizer, device, num_epochs)

training on  cuda
epoch 1, loss 1.9496, train acc 16.619, test acc 0.387, time 196.1 sec
epoch 2, loss 0.7395, train acc 14.424, test acc 0.478, time 195.4 sec
epoch 3, loss 0.4237, train acc 11.555, test acc 0.555, time 197.9 sec
epoch 4, loss 0.2793, train acc 9.603, test acc 0.615, time 196.7 sec
epoch 5, loss 0.2005, train acc 8.244, test acc 0.648, time 197.3 sec
epoch 6, loss 0.1512, train acc 7.240, test acc 0.665, time 203.6 sec
epoch 7, loss 0.1182, train acc 6.449, test acc 0.687, time 196.2 sec
epoch 8, loss 0.0943, train acc 5.860, test acc 0.692, time 196.4 sec
epoch 9, loss 0.0767, train acc 5.373, test acc 0.684, time 197.0 sec
epoch 10, loss 0.0632, train acc 4.973, test acc 0.683, time 192.8 sec
